# mom6-tools OO workflow demo

Demonstrates the `Case` obj oriented for inspecting and running diagnostics on a
MOM6 run — without touching the per-script CLIs.

Two entry paths are shown:
- **`Case.from_config()`** — the standard workflow; reads a `diag_config.yml`.
- **`Case.from_diag_table()`** — parses the model `diag_table` directly (no CESM config needed).

Two cases are covered:
1. **Regional** — `carib12_tides_runoff_test_mom6_tools`
2. **Global** — fill in `GLOBAL_CONFIG` / `GLOBAL_DIAG_TABLE` below

Run this notebook with the `mom6-tools` kernel:
```
conda activate /glade/work/ajanney/conda-envs/mom6-tools
jupyter lab
```

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

from mom6_tools import Case
from mom6_diagtables import parse_diag_table

## Paths

Edit the `GLOBAL_*` variables below when you have a global case ready.
The carib12 paths point at files already present in this repo / on Derecho.

In [11]:
REPO = Path("/glade/work/ajanney/Software/mom6-tools")

# --- Regional (carib12) -------------------------------------------------------
CARIB_CONFIG     = "/glade/work/ajanney/Software/mom6-tools/docs/example_configs_and_diags/diag_config.yml"
CARIB_DIAG_TABLE = Path("/glade/derecho/scratch/ajanney/carib12_tides_runoff_test_mom6_tools/run/diag_table")
CARIB_HIST_DIR   = Path("/glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/ocn/hist")
CARIB_GEOM       = None  # optional: set to Path("...ocean_geometry.nc") if land-block elimination is on

# --- Global (fill in) --------------------------------------------------------
GLOBAL_CONFIG     = Path("/glade/work/ajanney/Software/mom6-tools/docs/example_configs_and_diags/diag_config_global.yml")       # <-- set me
GLOBAL_DIAG_TABLE = Path("/glade/work/ajanney/Software/mom6-tools/docs/example_configs_and_diags/diag_table_global")   # <-- set me
GLOBAL_HIST_DIR   = Path("/glade/campaign/cgd/oce/projects/CROCODILE/model_output/b.e30_alpha08b.B1850C_LTso.ne30_t232_wgx3.328/ocn/hist/")               # <-- set me
GLOBAL_GEOM       = None  # optional: set to Path("...ocean_geometry.nc") if needed

---
## 1 — Regional case: `Case.from_config()`

The primary workflow: reads `diag_config.yml`, queries CESM for the run directory,
and builds the stream-to-glob mapping from the `Fnames` block.

In [12]:
carib = Case.from_config(CARIB_CONFIG)
carib

Case(DataSource(casename='carib12_tides_runoff_test_mom6_tools', outdir='/glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/ocn/hist/', streams=['geom', 'native', 'sfc', 'static', 'z']))

In [13]:
# What streams exist and which diagnostics can actually run?
carib.summary()

Case:   carib12_tides_runoff_test_mom6_tools
Outdir: /glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/ocn/hist/

Streams (5): geom, native, sfc, static, z

Diagnostics (2/2 runnable):
  moc                  ok
  surface              ok


In [ ]:
# Programmatic access to the runnable list
carib.vars

['moc', 'surface']

In [15]:
# Inspect the config that was loaded
carib.config

{'Case': {'CASEROOT': '/glade/work/ajanney/CESM-cases/carib12_tides_runoff_test_mom6_tools',
  'OCN_DIAG_ROOT': '/glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/hist/ocn',
  'SNAME': '130'},
 'Avg': {'start_date': '2000-01-01', 'end_date': '2001-12-31'},
 'Fnames': {'z': '.mom6.h.z.????-??.nc',
  'native': '.mom6.h.native.????-??.nc',
  'sfc': '.mom6.h.sfc.????-??.nc',
  'static': '.mom6.h.static.nc',
  'geom': '.mom6.h.ocean_geometry.nc'},
 'oce_cat': '/glade/u/home/gmarques/libs/oce-catalogs/reference-datasets.yml'}

---
## 2 — Regional case: `Case.from_diag_table()`

Alternative entry path: parse the model `diag_table` directly.
Stream names are inferred automatically from the file-prefix structure.

In [16]:
carib_dt = Case.from_diag_table(
    CARIB_DIAG_TABLE,
    outdir=CARIB_HIST_DIR,
    geom=CARIB_GEOM,   # None -> omitted; set above if land-block elimination is on
)
carib_dt.summary()

Case:   (unnamed)
Outdir: /glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/ocn/hist

Diag table: MOM6 diagnostic fields table for CESM case: carib12_tides_runoff_test_mom6_tools
  23 files, 221 fields
  h.rho2                 1 months   10 fields
  h.native               1 months   74 fields
  h.z                    1 months   15 fields
  h.sfc                  1 days     12 fields
  h.static              -1 days     20 fields
  Florida_Cuba           1 days     5 fields
  Florida_Strait         1 days     5 fields
  Yucatan_Peninsula      1 days     5 fields
  Windward_Passage       1 days     5 fields
  Mona_Passage           1 days     5 fields
  Central_Caribbean      1 days     5 fields
  WOCE_A22               1 days     5 fields
  Anegada_Passage        1 days     5 fields
  Lesser_Antilles        1 days     5 fields
  Antigua_Passage        1 days     5 fields
  Guadeloupe_Passage     1 days     5 fields
  Dominica_Passage       1 days     5 fields
  

In [17]:
# summary() includes the diag_table breakdown when available
carib_dt.summary()

Case:   (unnamed)
Outdir: /glade/derecho/scratch/ajanney/archive/carib12_tides_runoff_test_mom6_tools/ocn/hist

Diag table: MOM6 diagnostic fields table for CESM case: carib12_tides_runoff_test_mom6_tools
  23 files, 221 fields
  h.rho2                 1 months   10 fields
  h.native               1 months   74 fields
  h.z                    1 months   15 fields
  h.sfc                  1 days     12 fields
  h.static              -1 days     20 fields
  Florida_Cuba           1 days     5 fields
  Florida_Strait         1 days     5 fields
  Yucatan_Peninsula      1 days     5 fields
  Windward_Passage       1 days     5 fields
  Mona_Passage           1 days     5 fields
  Central_Caribbean      1 days     5 fields
  WOCE_A22               1 days     5 fields
  Anegada_Passage        1 days     5 fields
  Lesser_Antilles        1 days     5 fields
  Antigua_Passage        1 days     5 fields
  Guadeloupe_Passage     1 days     5 fields
  Dominica_Passage       1 days     5 fields
  

---
## 3 — Global case

Set `GLOBAL_CONFIG` and `GLOBAL_DIAG_TABLE` at the top of this notebook,
then re-run this section.

In [18]:
if GLOBAL_CONFIG.exists():
    globe = Case.from_config(GLOBAL_CONFIG)
    globe.summary()
else:
    print("GLOBAL_CONFIG not set — edit the Paths cell above and rerun.")

Case:   g.e30_a09b.GW_JAR.TL319_t201_wgx3_hycom1_N75.2026.001
Outdir: /glade/derecho/scratch/chengz/archive/g.e30_a09b.GW_JAR.TL319_t201_wgx3_hycom1_N75.2026.001/ocn/hist/

Streams (6): geom, native, rho2, sfc, static, z

Diagnostics (2/2 runnable):
  moc                  ok
  surface              ok


In [19]:
# Global via diag_table
if GLOBAL_DIAG_TABLE.exists():
    globe_dt = Case.from_diag_table(GLOBAL_DIAG_TABLE, outdir=GLOBAL_HIST_DIR, geom=GLOBAL_GEOM)
    globe_dt.summary()
else:
    print("GLOBAL_DIAG_TABLE not set — edit the Paths cell above and rerun.")

Case:   (unnamed)
Outdir: /glade/campaign/cgd/oce/projects/CROCODILE/model_output/b.e30_alpha08b.B1850C_LTso.ne30_t232_wgx3.328/ocn/hist

Diag table: MOM6 diagnostic fields table for CESM case: b.e30_alpha08b.B1850C_LTso.ne30_t232_wgx3.328
  24 files, 266 fields
  rho2                   1 months   11 fields
  native                 1 months   95 fields
  z                      1 months   20 fields
  sfc                    1 days     20 fields
  static                -1 days     25 fields
  Agulhas_Section        1 months   5 fields
  Barents_Opening        1 months   5 fields
  Bering_Strait          1 months   5 fields
  Bosphorus_Strait       1 months   5 fields
  Davis_Strait           1 months   5 fields
  Denmark_Strait         1 months   5 fields
  Drake_Passage          1 months   5 fields
  English_Channel        1 months   5 fields
  Fram_Strait            1 months   5 fields
  Florida_Bahamas_extended   1 months   5 fields
  Florida_Cuba           1 months   5 fields
  Gibral

---
## 4 — Running diagnostics

Diagnostic methods are dispatched through the registry.  `plot=False, save=False`
returns the raw `xarray` result without writing any files — useful for notebook exploration.

In [ ]:
# Single diagnostic — compute only, no cluster (nw=0).
# Returns an xarray Dataset you can inspect inline.
sfc = carib.surface(
    start_date="2000-01-01",
    end_date="2001-12-31",
    nw=4,
    plot=True,
    save=True,
)
sfc

Casename is: carib12_tides_runoff_test_mom6_tools
Number of workers to be used: 4
Requesting 4 workers... 



/glade/work/ajanney/conda-envs/mom6-tools-refactor/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38575 instead
  warnings.warn(


/proxy/38575/status
MOM6 grid successfully loaded... 

Reading surface dataset...
Time elapsed:  0:00:11.764450
Computing time mean and monthly climatology for: ['tos', 'SSU', 'SSV', 'speed', 'SSH']
Time elapsed:  0:00:15.136749

 Plotting...
Saving netCDF file: ncfiles/carib12_tides_runoff_test_mom6_tools_surface.nc


<xarray.Dataset> Size: 97MB
Dimensions:            (xh: 759, yh: 457, month: 12, xq: 760, yq: 458)
Coordinates:
  * xh                 (xh) float64 6kB -98.46 -98.38 -98.29 ... -35.62 -35.54
  * yh                 (yh) float64 4kB -5.958 -5.875 -5.792 ... 31.88 31.96
  * month              (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * xq                 (xq) float64 6kB -98.5 -98.42 -98.33 ... -35.58 -35.5
  * yq                 (yq) float64 4kB -6.0 -5.917 -5.834 ... 31.83 31.92 32.0
Data variables:
    mean_tos           (yh, xh) float64 3MB nan nan nan ... 23.39 23.29 23.08
    tos_climatology    (month, yh, xh) float32 17MB nan nan nan ... 22.16 21.97
    mean_SSU           (yh, xq) float64 3MB nan nan nan ... -0.06688 -0.05899
    SSU_climatology    (month, yh, xq) float32 17MB nan nan ... 0.03032 0.03834
    mean_SSV           (yq, xh) float64 3MB nan nan nan ... 0.1274 0.2601
    SSV_climatology    (month, yq, xh) float32 17MB nan nan ... 0.008482 0.09583
    mean_speed         (yh, xh) float64 3MB nan nan nan ... 0.3024 0.3097 0.4401
    speed_climatology  (month, yh, xh) float32 17MB nan nan ... 0.2802 0.4502
    mean_SSH           (yh, xh) float64 3MB nan nan nan ... 0.05725 0.07242
    SSH_climatology    (month, yh, xh) float32 17MB nan nan ... 0.0034 0.008859
Attributes:
    start_date:    2000-01-01
    end_date:      2001-12-31
    casename:      carib12_tides_runoff_test_mom6_tools
    description:   Surface-field time means and monthly climatologies
    module:        surface.py
    generated_by:  ajanney using mom6-tools, on 2026-06-10

In [46]:
# Global surface with a PBS cluster. get_cluster needs a project account:
# set it in the environment first, e.g.  os.environ['PBS_ACCOUNT'] = 'PXXXXXXXX'
sfc_global = globe_dt.surface(
    start_date="0001-01-01",
    end_date="0002-12-31",
    nw=6,
    plot=True,
    save=True,
)

Casename is: None
Number of workers to be used: 6
Requesting 6 workers... 

/proxy/8787/status
MOM6 grid successfully loaded... 

Reading surface dataset...
Time elapsed:  0:01:41.160284
Computing time mean and monthly climatology for: ['tos', 'SSU', 'SSV', 'speed', 'zos']
Time elapsed:  0:00:17.951524

 Plotting...
Saving netCDF file: ncfiles/_surface.nc


In [ ]:
# To reuse one cluster for
# several diagnostics. get_cluster reads PBS_ACCOUNT or pass account= via kwargs.
# from mom6_tools.core import Cluster
#
# with Cluster(nw=6) as cl:
#     sfc = carib.surface(start_date="2000-01-01", end_date="2001-12-31",
#                         nw=6, plot=True, save=True)

In [ ]:
# Run everything that can run for this case (skips diagnostics with missing streams)
# results = carib.run_all(nw=0, plot=False, save=False)
# list(results)   # -> ['moc', ...] depending on registered diagnostics

---
## 5 — Explicit file paths: `Case.from_files()`

When you have files in hand and don't need CESM config resolution.

In [ ]:
manual = Case.from_files(
    casename="my_run",
    outdir=str(CARIB_HIST_DIR),
    z="carib12_tides_runoff_test_mom6_tools.mom6.h.z*.nc",
    native="carib12_tides_runoff_test_mom6_tools.mom6.h.native*.nc",
    static="carib12_tides_runoff_test_mom6_tools.mom6.h.static.nc",
    geom="carib12_tides_runoff_test_mom6_tools.mom6.h.ocean_geometry.nc",
)
manual.summary()

---
## 6 — Parsing a diag_table standalone

`mom6-diagtables` is usable independently of the `Case` workflow.

In [ ]:
# Example tables shipped under docs/example_configs_and_diags/
global_table  = parse_diag_table(REPO / "docs/example_configs_and_diags/diag_table_global")
carib_table   = parse_diag_table(REPO / "docs/example_configs_and_diags/diag_table_regional_carib")

print("Global:",  len(global_table.files),  "files,", len(global_table.fields),  "fields")
print("Carib:",   len(carib_table.files),   "files,", len(carib_table.fields),   "fields")

In [ ]:
# Stream names (default: common-prefix heuristic; pass pattern=/mapping= to override)
print("Global streams:",  sorted(global_table.streams()))
print("Carib streams:",   sorted(carib_table.streams()))

In [ ]:
# Fields written to the 'z' stream in the global table
z_file = global_table.streams()["z"]
z_fields = global_table.fields_for(z_file.file_name)
print(f"'z' stream: {len(z_fields)} fields")
[f.field_name for f in z_fields[:10]]   # first 10

In [ ]:
# YAML diag_table works the same way
yaml_table = parse_diag_table(REPO / "packages/mom6-diagtables/tests/data/diag_table_global.yaml")
print("YAML table:", len(yaml_table.files), "files,", len(yaml_table.fields), "fields")
yaml_table.streams()